In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


# Deep Dive into PyTorch Autograd: The Engine of Neural Networks

### 1. What is Autograd?
Autograd stands for **Automatic Differentiation**. In deep learning, we need to calculate gradients (derivatives) of a loss function with respect to model parameters (weights) to perform optimization.

- **Manual Calculus**: Calculating derivatives by hand is prone to error and impossible for deep architectures.
- **Autograd**: PyTorch records all operations performed on tensors and automatically computes the gradients using the chain rule.

### 2. The Intuition: Computational Graphs
PyTorch builds a **Directed Acyclic Graph (DAG)** dynamically as you write code.
- **Nodes**: Represent Tensors.
- **Edges**: Represent functions or operations (e.g., addition, exponentiation).

When we call `.backward()`, PyTorch traverses this graph from the output node back to the input nodes, multiplying gradients along the way (Chain Rule).

In [1]:
import torch

# Step 1: Create a tensor and set requires_grad=True to track computation
x = torch.tensor(3.0, requires_grad=True)

# Step 2: Define a function y = x^3 + 10
y = x**3 + 10

# Step 3: Compute gradients
y.backward()

# The derivative dy/dx = 3x^2. For x=3, 3*(3^2) = 27
print(f"Input x: {x.item()}")
print(f"Output y: {y.item()}")
print(f"Gradient (dy/dx) at x=3: {x.grad}")

Input x: 3.0
Output y: 37.0
Gradient (dy/dx) at x=3: 27.0


# Beginner's Guide to Implementing Autograd

To use Autograd, we follow a specific sequence of steps. Let's build a simple neuron-like calculation: $Output = (Weight \times Input) + Bias$.

### Step 1: Define Input and Parameters
We need to tell PyTorch which variables are 'learnable' (meaning we want to find their gradients).

In [4]:
import torch

# 'x' is our input (data). We usually don't need gradients for inputs.
x = torch.tensor(2.0)

# 'w' and 'b' are parameters. We set requires_grad=True because
# we want PyTorch to calculate how changing them affects the result.
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

print(f"Initial Weight w: {w}")
print(f"Initial Bias b: {b}")

Initial Weight w: 3.0
Initial Bias b: 1.0


### Step 2: The Forward Pass (Building the Graph)
When you perform math with these tensors, PyTorch internally creates a 'Computational Graph'. It remembers that `y` was created using multiplication and addition.

In [5]:
# This is the forward pass calculation
y = w * x + b

print(f"Result y: {y}")
# Look at 'grad_fn' - it shows the operation that created y (AddBackward)
print(f"Operation tracking: {y.grad_fn}")

Result y: 7.0
Operation tracking: <AddBackward0 object at 0x7b9d49704910>


### Step 3: The Backward Pass (Calculating Gradients)
When we call `.backward()`, PyTorch goes through the graph in reverse. It calculates the derivative of `y` with respect to `w` and `b` automatically.

In [6]:
# Calculate gradients
y.backward()

# Now, the '.grad' attributes of w and b are populated
# dy/dw = x (which is 2.0)
# dy/db = 1.0
print(f"Gradient of w: {w.grad}")
print(f"Gradient of b: {b.grad}")

Gradient of w: 2.0
Gradient of b: 1.0


### Step 4: Why do we do this?
In machine learning, if our `y` was a 'Loss' (error), the gradients tell us:
1. **Direction**: Should we increase or decrease the weight?
2. **Magnitude**: How much does the error change if we change the weight by a tiny bit?

**Note on Zeroing Gradients:**
By default, PyTorch **accumulates** (adds up) gradients. If you run the backward pass again, it will add the new gradients to the old ones. In training loops, we always clear them using `w.grad.zero_()` before the next step.

# Real-World Use Case: Training a Linear Regression Model

In this example, we have some data points and we want our model to learn the relationship $y = 2x$. We will use Autograd to find the best weight automatically.

### 1. Create Synthetic Data
We create inputs `x` and target outputs `y_target`.

In [7]:
import torch
import matplotlib.pyplot as plt

# Inputs (x) and Targets (y = 2x)
x_train = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_target = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

# Our model parameter (the weight we want to learn)
# We initialize it to a random value
w = torch.tensor([[0.0]], requires_grad=True)

print(f"Starting weight: {w.item()}")

Starting weight: 0.0


### 2. The Training Loop
In every step (epoch):
1. **Predict**: Calculate output using current weight.
2. **Loss**: Calculate the Mean Squared Error.
3. **Backward**: Use Autograd to find the gradient.
4. **Update**: Move the weight in the opposite direction of the gradient (Gradient Descent).



Quick Summary: PyTorch Autograd Implementation

If you want to implement Autograd in any project, follow these **4 mandatory steps**:

### 1. Initialize Tensors with Tracking
Any variable you want to optimize (Weights/Bias) must have `requires_grad=True`.
```python
import torch
x = torch.tensor(1.0) # Input (No tracking)
w = torch.tensor(2.0, requires_grad=True) # Parameter (Tracking ON)
```

### 2. The Forward Pass
Perform your math normally. PyTorch builds the graph automatically.
```python
loss = (x * w) - 10 # Example operation
```

### 3. The Backward Pass
Trigger the calculus engine to find the derivatives.
```python
loss.backward()
print(w.grad) # Access the calculated gradient
```

### 4. Zero the Gradients
**Crucial Step:** Gradients accumulate by default. Always clear them before the next calculation.
```python
w.grad.zero_()
```

In [9]:
# Complete Template Code for Quick Implementation
import torch

# 1. Setup Data and Parameters
x = torch.tensor([1.0, 2.0])
w = torch.randn(2, requires_grad=True)

# 2. Forward Pass (Math)
prediction = (x * w).sum()
loss = (prediction - 5)**2

# 3. Backward Pass (Calculus)
loss.backward()

# 4. Use and Clear
print(f"Weight Gradient: {w.grad}")

# Manual Update Example (Gradient Descent)
with torch.no_grad():
    w -= 0.01 * w.grad

# Always clear for the next loop
w.grad.zero_()

print("Implementation Complete!")

Weight Gradient: tensor([ -5.9867, -11.9735])
Implementation Complete!


In [8]:
learning_rate = 0.01

for epoch in range(100):
    # 1. Forward Pass: Predict y
    y_pred = x_train @ w

    # 2. Compute Loss (Mean Squared Error)
    loss = torch.mean((y_pred - y_target)**2)

    # 3. Backward Pass: Autograd calculates dLoss/dw
    loss.backward()

    # 4. Update Weights using Gradient Descent
    with torch.no_grad():
        w -= learning_rate * w.grad

    # 5. VERY IMPORTANT: Zero the gradients for the next round
    w.grad.zero_()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, Weight = {w.item():.4f}")

print(f"\nFinal Learned Weight: {w.item():.4f} (Target was 2.0)")

Epoch 20: Loss = 0.0624, Weight = 1.9225
Epoch 40: Loss = 0.0001, Weight = 1.9970
Epoch 60: Loss = 0.0000, Weight = 1.9999
Epoch 80: Loss = 0.0000, Weight = 2.0000
Epoch 100: Loss = 0.0000, Weight = 2.0000

Final Learned Weight: 2.0000 (Target was 2.0)


### 3. Key Attributes

- **`requires_grad`**: If True, PyTorch starts tracking all operations on this tensor.
- **`grad_fn`**: Stores the operation that created the tensor (e.g., `MulBackward`, `AddBackward`).
- **`is_leaf`**: Tensors created by the user (not by an operation) are leaf tensors.
- **`.backward()`**: Initiates the backpropagation process.
- **`.grad`**: The attribute where the computed gradient is stored.

In [2]:
# Complex example with multiple tensors
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(5.0)

# Linear model: z = w*x + b
z = w * x + b

# Loss function: L = (z - 10)^2
loss = (z - 10)**2

loss.backward()

# dL/dw = 2(z-10) * x
# dL/db = 2(z-10) * 1
print(f"Gradient w.r.t weights (w): {w.grad}")
print(f"Gradient w.r.t bias (b): {b.grad}")
print(f"Gradient w.r.t input (x): {x.grad}") # None, because requires_grad was False

Gradient w.r.t weights (w): -30.0
Gradient w.r.t bias (b): -6.0
Gradient w.r.t input (x): None


### 4. Real-World Use Case & Comparison

**Real-Life Analogy**: Imagine you are fine-tuning a recipe. You change the salt (weight). The 'Autograd' system tells you exactly how much the 'Saltiness' (Loss) changes based on that tiny adjustment, so you know whether to add more or less.

**Comparison: Autograd vs. Numerical Differentiation**
- **Numerical**: Requires evaluating the function multiple times ($f(x+h) - f(x) / h$). It is an approximation and slow.
- **Autograd (Analytic)**: Computes the exact derivative in one pass through the graph. It is the gold standard for speed and precision.

### 5. Stopping Autograd
During inference (testing), we don't need gradients. Disabling Autograd saves memory and compute power.

In [3]:
# Two ways to disable tracking
with torch.no_grad():
    val = x * 10
    print(f"Tracking enabled? {val.requires_grad}")

# Using detach to create a copy that is no longer tracked
detached_w = w.detach()
print(f"Detached tensor tracking? {detached_w.requires_grad}")

Tracking enabled? False
Detached tensor tracking? False
